In [1]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH       = '/kaggle/input/datasets/ankitdhiman7/race-dataset'
PROCESSED_PATH = '/kaggle/working/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# Load only train.csv since all files are identical
df = pd.read_csv(f'{RAW_PATH}/train.csv')
print(f'Full dataset: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Full dataset: (87866, 9)
Columns: ['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer']


In [2]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

text_columns = ['article', 'question', 'A', 'B', 'C', 'D']
for col in text_columns:
    df[col] = df[col].apply(clean_text)

df.dropna(subset=text_columns, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'After cleaning: {df.shape}')

After cleaning: (87866, 9)


In [3]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
dev_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df.reset_index(drop=True, inplace=True)
dev_df.reset_index(drop=True,   inplace=True)
test_df.reset_index(drop=True,  inplace=True)

print(f'Train: {train_df.shape}')
print(f'Dev:   {dev_df.shape}')
print(f'Test:  {test_df.shape}')

# Save clean splits
train_df.to_csv(f'{PROCESSED_PATH}/train_clean.csv', index=False)
dev_df.to_csv(f'{PROCESSED_PATH}/dev_clean.csv',     index=False)
test_df.to_csv(f'{PROCESSED_PATH}/test_clean.csv',   index=False)

Train: (70292, 9)
Dev:   (8787, 9)
Test:  (8787, 9)


In [4]:
def build_option_dataset(df):
    rows = []
    for _, row in df.iterrows():
        for opt in ['A', 'B', 'C', 'D']:
            combined = f"{row['article']} {row['question']} {row[opt]}"
            label    = 1 if row['answer'] == opt else 0
            rows.append({
                'combined_text': combined,
                'article':       row['article'],
                'question':      row['question'],
                'option':        row[opt],
                'option_label':  opt,
                'label':         label
            })
    return pd.DataFrame(rows)

print('Building option datasets')
train_options = build_option_dataset(train_df)
dev_options   = build_option_dataset(dev_df)
test_options  = build_option_dataset(test_df)

print(f'Train options: {train_options.shape}')
print(f'Dev options:   {dev_options.shape}')
print(f'Test options:  {test_options.shape}')
print(f'Class balance: {train_options["label"].mean():.2f}')

Building option datasets
Train options: (281168, 6)
Dev options:   (35148, 6)
Test options:  (35148, 6)
Class balance: 0.25


In [5]:
train_options.to_csv(f'{PROCESSED_PATH}/train_options.csv', index=False)
dev_options.to_csv(f'{PROCESSED_PATH}/dev_options.csv',     index=False)
test_options.to_csv(f'{PROCESSED_PATH}/test_options.csv',   index=False)

print(os.listdir(PROCESSED_PATH))

['dev_options.csv', 'train_clean.csv', 'dev_clean.csv', 'train_options.csv', 'test_clean.csv', 'test_options.csv']
